In [1]:
from __future__ import annotations

import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import wandb
from scipy.signal import find_peaks
from tqdm.autonotebook import tqdm


C:\Users\Colin Leong\AppData\Local\Temp\ipykernel_16636\2360345431.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### File-parsing function

In [2]:
def parse_file(df: pd.DataFrame, column_to_parse="file") -> pd.DataFrame:
    df["file_stem"] = df[column_to_parse].apply(lambda f: Path(f).stem.split(".")[0])
    df["file_stem"] = df["file_stem"].str.replace(
        "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_", ""
    )
    df["cbt_number"] = df["file_stem"].apply(lambda f: int(f.split("-")[1]) if f.lower().startswith("cbt") else None)
    video_types = ["Intro", "Passage", "More Info"]
    df["cbt_video_type"] = df["file_stem"].apply(
        lambda f: next((v for v in video_types if v.lower() in f.lower()), None)
    )

    # Video title, e.g. "God Creates The World"
    df["title"] = df["file_stem"].apply(
        lambda f: f.split("_")[1].strip().title() if f.lower().startswith("cbt") else None
    )
    df["title"] = df["title"].astype("string")
    return df

### Load Video Window-Embeddings


#### Uncomment to create hive-format data

In [3]:
# window_embeddings_folder = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\chronological_bible_translation_in_american_sign_language_119_introductions_and_passages")
# import shutil

# children = list(window_embeddings_folder.glob("*"))
# out_dir = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\hive_format")
# out_dir.mkdir(exist_ok=True, parents=True)
# for child in children:
#     child_name = child.name
#     print(child_name)
#     # child.rename(ch)
#     new_name = child_name.replace(
#         "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_", ""
#     )
#     new_name = child_name.replace(".pose-mediapipe.window_embeddings", "")
#     new_name = f"video_stem={new_name}"
#     print(new_name)
#     new_path = out_dir / new_name
#     print(new_path)
#     shutil.copytree(child, new_path)

In [75]:

window_embeddings_folder = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\hive_format")

In [76]:
dataset = ds.dataset(window_embeddings_folder, format="parquet",partitioning="hive")
dataset.count_rows()
# df = ds.count_rows
dataset.schema

window_start_ms: int64
window_end_ms: int64
window_midpoint_ms: int64
embedding: list<element: double>
  child 0, element: double
video_stem: string
model: string
window_size_ms: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 817

In [77]:
# window_embeddings_df = dataset.to_table().to_pandas()

ds_filter = (
    # (ds.field("model") == "asl_finetune_checkpoint_best")
    (ds.field("window_size_ms") <= 900) & (ds.field("window_size_ms") > 600)
)
# ds_filter = ((ds.field("dataset")=="asl-citizen") & (ds.field("embedding_model")=="asl_finetune_checkpoint_best"))
window_embeddings_df = dataset.to_table(filter=ds_filter).to_pandas()

In [78]:
window_embeddings_df["file"] = window_embeddings_df["video_stem"].str.replace(
    "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_cbt", "CBT"
)
window_embeddings_df["file"] = window_embeddings_df["file"].str.replace("passage", "Passage")

In [79]:
window_embeddings_df = parse_file(window_embeddings_df)
window_embeddings_df["title"] = window_embeddings_df["title"].str.title()

In [108]:
window_embeddings_df['title'].unique()

<StringArray>
[                        'God Creates The World',
                             'Abram Rescues Lot',
                'God Gives His Ten Commandments',
                    'Israel Fails To Conquer Ai',
                           'David Faces Goliath',
 'The Chosen Servant Of God Will Face Suffering',
                            'The Birth Of Jesus',
              'Jesus Explains How To Rebuke Sin',
                 'Jesus Is Crucified And Buried',
                       'Paul Travels To Corinth']
Length: 10, dtype: string

In [109]:
window_embeddings_df['cbt_number'].unique()

array([  1,  10,  26,  33,  39,  43,  46,  64,  79, 104])

In [80]:
window_embeddings_df.head()

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title
0,0,650,325,"[0.253972589969635, -0.3248118758201599, -0.03...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,650,CBT-001-ase-3-Passage _ god creates the world,CBT-001-ase-3-Passage _ god creates the world,1,Passage,God Creates The World
1,100,750,425,"[0.253972589969635, -0.3248118758201599, -0.03...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,650,CBT-001-ase-3-Passage _ god creates the world,CBT-001-ase-3-Passage _ god creates the world,1,Passage,God Creates The World
2,200,850,525,"[0.24460630118846893, -0.31938108801841736, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,650,CBT-001-ase-3-Passage _ god creates the world,CBT-001-ase-3-Passage _ god creates the world,1,Passage,God Creates The World
3,300,950,625,"[0.253972589969635, -0.3248118758201599, -0.03...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,650,CBT-001-ase-3-Passage _ god creates the world,CBT-001-ase-3-Passage _ god creates the world,1,Passage,God Creates The World
4,400,1050,725,"[0.24460630118846893, -0.31938108801841736, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,650,CBT-001-ase-3-Passage _ god creates the world,CBT-001-ase-3-Passage _ god creates the world,1,Passage,God Creates The World


In [81]:
window_embeddings_df["window_size_ms"].unique()


array([650, 700, 750, 800, 900], dtype=int32)

### What embedding window sizes are available?

In [82]:
collector = []
for window_size_ms in window_embeddings_df["window_size_ms"].unique():
    window_df = window_embeddings_df[window_embeddings_df["window_size_ms"]==window_size_ms]
    title_count = len(window_df['title'].unique())
    print(f"{window_size_ms}: {len(window_df)} rows, {len(window_df['title'].unique())}")
    if title_count == 10:
        # print(f"Keeping {window_size_ms} ms")
        collector.append(window_df)

window_embeddings_df = pd.concat(collector)

650: 57504 rows, 10
700: 57500 rows, 10
750: 57494 rows, 10
800: 57490 rows, 10
900: 57480 rows, 10


In [83]:
window_embeddings_df["window_size_ms"].unique()

array([650, 700, 750, 800, 900], dtype=int32)

In [84]:
window_embeddings_df["title"].unique()

<StringArray>
[                        'God Creates The World',
                             'Abram Rescues Lot',
                'God Gives His Ten Commandments',
                    'Israel Fails To Conquer Ai',
                           'David Faces Goliath',
 'The Chosen Servant Of God Will Face Suffering',
                            'The Birth Of Jesus',
              'Jesus Explains How To Rebuke Sin',
                 'Jesus Is Crucified And Buried',
                       'Paul Travels To Corinth']
Length: 10, dtype: string

In [85]:
window_embeddings_300_ms = window_embeddings_df[window_embeddings_df["window_size_ms"] ==300]

In [86]:
window_embeddings_300_ms

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title


### Load Ground Truth


In [87]:
osb_glosses_version="1_4"
osb_glosses_dir = Path(r"C:\Users\Colin Leong\projects\SignBiblesDatasetPaper\Avodah_Annotations") / f"osb_glosses_v{osb_glosses_version}" /"osb_glosses"
osb_glosses_dfs = []

for truth_file in osb_glosses_dir.rglob("*.tsv"):
    truth_file_df = pd.read_csv(truth_file, delimiter="\t")
    # truth_file_df['video_title'] = truth_file.stem # alread included.
    truth_file_df = parse_file(truth_file_df)
    osb_glosses_dfs.append(truth_file_df)
osb_glosses_truth_df = pd.concat(osb_glosses_dfs)
osb_glosses_truth_df["window_midpoint_ms"] = osb_glosses_truth_df["midpoint_sec"] * 1000
osb_glosses_truth_df = osb_glosses_truth_df.rename(columns={"Sign":"Original Sign","ASL Citizen Gloss":"Sign"})
osb_glosses_truth_df.head()

,Original Sign,Begin Time - hh:mm:ss.ms,Begin Time - ss.msec,End Time - hh:mm:ss.ms,End Time - ss.msec,file,duration_sec,midpoint_sec,Time (minutes:sec),Sign,ASL Citizen Gloss Repeat Count,file_stem,cbt_number,cbt_video_type,title,window_midpoint_ms
0,EARTH,00:00:06.410,6.41,00:00:06.520,6.52,CBT-001-ase-3-Passage _ God Creates the World,0.11,6.465,0:06,EARTH,1,CBT-001-ase-3-Passage _ God Creates the World,1,Passage,God Creates The World,6465.0
1,GOD,00:00:07.770,7.77,00:00:08.190,8.19,CBT-001-ase-3-Passage _ God Creates the World,0.42,7.980,0:07,GOD,1,CBT-001-ase-3-Passage _ God Creates the World,1,Passage,God Creates The World,7980.0
2,CREATE,00:00:08.620,8.62,00:00:09.850,9.85,CBT-001-ase-3-Passage _ God Creates the World,1.23,9.235,0:09,CREATE,1,CBT-001-ase-3-Passage _ God Creates the World,1,Passage,God Creates The World,9235.0
3,DAY,00:00:14.400,14.40,00:00:14.920,14.92,CBT-001-ase-3-Passage _ God Creates the World,0.52,14.660,0:14,DAY,1,CBT-001-ase-3-Passage _ God Creates the World,1,Passage,God Creates The World,14660.0
4,GOD,00:00:15.620,15.62,00:00:15.730,15.73,CBT-001-ase-3-Passage _ God Creates the World,0.11,15.675,0:15,GOD,1,CBT-001-ase-3-Passage _ God Creates the World,1,Passage,God Creates The World,15675.0


In [88]:
osb_glosses_truth_df[osb_glosses_truth_df["Sign"]=="BAR"]

,Original Sign,Begin Time - hh:mm:ss.ms,Begin Time - ss.msec,End Time - hh:mm:ss.ms,End Time - ss.msec,file,duration_sec,midpoint_sec,Time (minutes:sec),Sign,ASL Citizen Gloss Repeat Count,file_stem,cbt_number,cbt_video_type,title,window_midpoint_ms
112,BAR,00:09:27.240,567.24,00:09:27.840,567.84,CBT-033-ase-2-Passage _ Israel Fails to Conque...,0.60,567.54,9:27,BAR,1,CBT-033-ase-2-Passage _ Israel Fails to Conque...,33,Passage,Israel Fails To Conquer Ai,567540.0
126,BAR,00:10:29.190,629.19,00:10:30.010,630.01,CBT-033-ase-2-Passage _ Israel Fails to Conque...,0.82,629.60,10:29,BAR,1,CBT-033-ase-2-Passage _ Israel Fails to Conque...,33,Passage,Israel Fails To Conquer Ai,629600.0


In [89]:
osb_glosses_truth_df[osb_glosses_truth_df["Sign"]=="BAR"]["duration_sec"].unique()

array([0.6 , 0.82])

In [90]:
len(osb_glosses_truth_df)

681

In [91]:
len(osb_glosses_truth_df["Sign"].unique())

93

### Load isolated embeddings

In [92]:
isolated_sign_embeddings_dir = Path(r"C:\Users\Colin Leong\data\isolated_sign_embeddings\embeddings")
dataset = ds.dataset(isolated_sign_embeddings_dir, format="parquet",partitioning="hive")
ds_filter = ((ds.field("dataset")=="asl-citizen") & (ds.field("embedding_model")=="asl_finetune_checkpoint_best"))
isolated_sign_embeddings_df = dataset.to_table(filter=ds_filter).to_pandas()

In [93]:
isolated_sign_embeddings_df.head()

,VIDEO_ID,GLOSS,POSE_FILE_PATH,VIDEO_FILE_PATH,embedding,dataset,SPLIT,embedding_model
0,000017451997373907346,LIBRARY,/opt/home/cleong/data/ASL_Citizen/poses/pose/0...,nan,"[0.081679806, -0.12799999, -0.08922268, 0.0277...",asl-citizen,test,asl_finetune_checkpoint_best
1,0000197996356050556,CELERY,/opt/home/cleong/data/ASL_Citizen/poses/pose/0...,nan,"[0.24444945, 0.19033322, 0.506465, 0.14141789,...",asl-citizen,test,asl_finetune_checkpoint_best
2,00010630150123391857,CANDY1,/opt/home/cleong/data/ASL_Citizen/poses/pose/0...,nan,"[-0.3333069, 0.18183184, 0.20270634, 0.1830834...",asl-citizen,test,asl_finetune_checkpoint_best
3,00012571487478130194,CASTLE2,/opt/home/cleong/data/ASL_Citizen/poses/pose/0...,nan,"[-0.31474665, -0.16275023, -0.06034531, 0.0231...",asl-citizen,test,asl_finetune_checkpoint_best
4,00015575668519507424,GOVERNMENT,/opt/home/cleong/data/ASL_Citizen/poses/pose/0...,nan,"[-0.075292826, -0.22823307, 0.29996705, -0.103...",asl-citizen,test,asl_finetune_checkpoint_best


### Load Transcripts

In [94]:
import json

In [95]:
transcripts_dir = Path(
    r"C:\Users\Colin Leong\data\DBL\webdataset_extracted_transcripts_only\transcripts\data\petabyte\cleong\data\DBL_Deaf_Bibles\webdataset_extracted\ase\chronological_bible_translation_in_american_sign_language_119_introductions_and_passages"
)


transcripts_collector = []
for transcript_file in transcripts_dir.glob("*.json"):
    print(transcript_file.name)
    with open(transcript_file, "r") as f:
        transcripts_for_file = json.load(f)
        print(transcripts_for_file)
        transcripts_for_file_df = pd.DataFrame.from_records(transcripts_for_file)
        transcripts_for_file_df["file"] = transcript_file.name
        transcripts_collector.append(transcripts_for_file_df)
transcripts_df = pd.concat(transcripts_collector)
transcripts_df = parse_file(transcripts_df)

ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_cbt-001-ase-2-intro _ god creates the world.transcripts.json
[{'text': 'In the beginning God created the heavens and the earth. Now the earth was formless and void, and darkness was over the surface of the deep. And the Spirit of God was hovering over the surface of the waters. And God said, “Let there be light,” and there was light. And God saw that the light was good, and He separated the light from the darkness. God called the light “day,” and the darkness He called “night.” And there was evening, and there was morning—the first day. And God said, “Let there be an expanse between the waters, to separate the waters from the waters.” So God made the expanse and separated the waters beneath it from the waters above. And it was so. God called the expanse “sky.” And there was evening, and there was morning—the second day. And God said, “Let the waters under the sky be gathered into one place, so 

In [96]:
transcripts_df

,text,start_frame,end_frame,language,license,source,biblenlp-vref,bible-ref,file,file_stem,cbt_number,cbt_video_type,title
0,In the beginning God created the heavens and t...,0,7876,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","GEN 1:1-31,2:1-3",ase_chronological_bible_translation_in_america...,cbt-001-ase-2-intro _ god creates the world,1,Intro,God Creates The World
0,In the beginning God created the heavens and t...,30,1359,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[0, 1, 2, 3, 4]",Genesis 1:1-5,ase_chronological_bible_translation_in_america...,cbt-001-ase-3-passage _ god creates the world,1,Passage,God Creates The World
1,"And God said, “Let there be an expanse between...",1360,1939,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[5, 6, 7]",Genesis 1:6-8,ase_chronological_bible_translation_in_america...,cbt-001-ase-3-passage _ god creates the world,1,Passage,God Creates The World
2,"And God said, “Let the waters under the sky be...",1940,3309,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[8, 9]",Genesis 1:9-10,ase_chronological_bible_translation_in_america...,cbt-001-ase-3-passage _ god creates the world,1,Passage,God Creates The World
3,"And God said, “Let there be lights in the expa...",3310,4519,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[13, 14, 15, 16, 17, 18]",Genesis 1:14-19,ase_chronological_bible_translation_in_america...,cbt-001-ase-3-passage _ god creates the world,1,Passage,God Creates The World
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,And all who dwell on the earth will worship th...,5330,6599,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,[30984],Revelation 13:8,ase_chronological_bible_translation_in_america...,cbt-119-ase-3-passage _ the beast from the sea...,119,Passage,The Beast From The Sea And The Beast From The ...
4,"He who has an ear, let him hear: “If anyone is...",6600,7419,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[30985, 30986]",Revelation 13:9-10,ase_chronological_bible_translation_in_america...,cbt-119-ase-3-passage _ the beast from the sea...,119,Passage,The Beast From The Sea And The Beast From The ...
5,Then I saw another beast rising out of the ear...,7420,9719,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[30987, 30988, 30989, 30990]",Revelation 13:11-14,ase_chronological_bible_translation_in_america...,cbt-119-ase-3-passage _ the beast from the sea...,119,Passage,The Beast From The Sea And The Beast From The ...
6,The second beast was permitted to give breath ...,9720,11279,"{'name': 'English', 'ISO639-3': 'eng', 'BCP-47...",public domain,http://ebible.org/engbsb/,"[30991, 30992, 30993]",Revelation 13:15-17,ase_chronological_bible_translation_in_america...,cbt-119-ase-3-passage _ the beast from the sea...,119,Passage,The Beast From The Sea And The Beast From The ...


### Pick a video and ground truth and search for glosses

#### Looking for BAR 
We gotta find truth annotations with embeddings!

In [97]:
potential_queries_df = osb_glosses_truth_df[osb_glosses_truth_df["Sign"]=="BAR"].copy()
potential_queries_df = potential_queries_df[potential_queries_df["duration_sec"]>0.2]
potential_queries_df = potential_queries_df[potential_queries_df["duration_sec"]<0.9]
potential_queries_df.head()

,Original Sign,Begin Time - hh:mm:ss.ms,Begin Time - ss.msec,End Time - hh:mm:ss.ms,End Time - ss.msec,file,duration_sec,midpoint_sec,Time (minutes:sec),Sign,ASL Citizen Gloss Repeat Count,file_stem,cbt_number,cbt_video_type,title,window_midpoint_ms
112,BAR,00:09:27.240,567.24,00:09:27.840,567.84,CBT-033-ase-2-Passage _ Israel Fails to Conque...,0.60,567.54,9:27,BAR,1,CBT-033-ase-2-Passage _ Israel Fails to Conque...,33,Passage,Israel Fails To Conquer Ai,567540.0
126,BAR,00:10:29.190,629.19,00:10:30.010,630.01,CBT-033-ase-2-Passage _ Israel Fails to Conque...,0.82,629.60,10:29,BAR,1,CBT-033-ase-2-Passage _ Israel Fails to Conque...,33,Passage,Israel Fails To Conquer Ai,629600.0


#### Two instances exist! 

##### 567.24 to 567.84 seconds (0.6s)

In [98]:
potential_queries_df.iloc[0]

Original Sign                                                                   BAR
Begin Time - hh:mm:ss.ms                                               00:09:27.240
Begin Time - ss.msec                                                         567.24
End Time - hh:mm:ss.ms                                                 00:09:27.840
End Time - ss.msec                                                           567.84
file                              CBT-033-ase-2-Passage _ Israel Fails to Conque...
duration_sec                                                                    0.6
midpoint_sec                                                                 567.54
Time (minutes:sec)                                                             9:27
Sign                                                                            BAR
ASL Citizen Gloss Repeat Count                                                    1
file_stem                         CBT-033-ase-2-Passage _ Israel Fails to Co

This one requires a window that encompasses  567.24 sec and 567.84 sec, a duration of 0.6 seconds

##### 629.19 to 630.01 seconds (0.82s)

In [99]:
potential_queries_df.iloc[1]

Original Sign                                                                   BAR
Begin Time - hh:mm:ss.ms                                               00:10:29.190
Begin Time - ss.msec                                                         629.19
End Time - hh:mm:ss.ms                                                 00:10:30.010
End Time - ss.msec                                                           630.01
file                              CBT-033-ase-2-Passage _ Israel Fails to Conque...
duration_sec                                                                   0.82
midpoint_sec                                                                  629.6
Time (minutes:sec)                                                            10:29
Sign                                                                            BAR
ASL Citizen Gloss Repeat Count                                                    1
file_stem                         CBT-033-ase-2-Passage _ Israel Fails to Co

This is an embedding that encompasses  629.19 sec and 630.01 sec, a duration of 0.82 seconds

So if we want one to search the other we should probably get an embedding size of 850 or 900 ms.

Let's use 1 as query and the other as target. 

In [100]:
window_embeddings_df["window_size_ms"].unique()

array([650, 700, 750, 800, 900], dtype=int32)

Let's load the 900 ms embeddings

In [111]:
window_embeddings_df['cbt_number'].unique()

array([  1,  10,  26,  33,  39,  43,  46,  64,  79, 104])

In [114]:
potential_target_df = window_embeddings_df[
    (window_embeddings_df["cbt_number"] == 33) & (window_embeddings_df["window_size_ms"] == 900)
].copy()
potential_target_df.head(2)

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title
98574,0,900,450,"[0.18340295553207397, -0.2988400161266327, -0....",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
98575,100,1000,550,"[0.18340295553207397, -0.2988400161266327, -0....",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai


In [ ]:
# 567.24 sec and 567.84 (0.6 seconds)
potential_query_embeddings = potential_target_df[potential_target_df["window_start_ms"]>567*1000]
potential_query_embeddings = potential_query_embeddings[potential_query_embeddings["window_end_ms"]<569*1000]
potential_query_embeddings.head()

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title
104245,567100,568000,567550,"[-0.026185424998402596, -0.08117301762104034, ...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
104246,567200,568100,567650,"[-0.009068762883543968, -0.11251287907361984, ...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
104247,567300,568200,567750,"[0.17952515184879303, -0.15540680289268494, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
104248,567400,568300,567850,"[0.20719218254089355, -0.23178017139434814, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
104249,567500,568400,567950,"[0.18852883577346802, -0.24942615628242493, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai


        window_start_ms	window_end_ms	window_midpoint_ms	
104246	567200	568100	567650
That should work, maybe

In [126]:
# 629.19 sec and 630.01 sec, a duration of 0.82 seconds
potential_query_embeddings = potential_target_df[potential_target_df["window_start_ms"] >= (629 * 1000)]
potential_query_embeddings = potential_query_embeddings[
    potential_query_embeddings["window_end_ms"] <= ((630.01) * 1000)
]
potential_query_embeddings.head()

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title
104864,629000,629900,629450,"[0.15811368823051453, 0.029370475560426712, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai
104865,629100,630000,629550,"[0.18030354380607605, -0.08589434623718262, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai


104865	629100	630000
^ that outta work

In [127]:
potential_query_embeddings.loc[104865]

window_start_ms                                                  629100
window_end_ms                                                    630000
window_midpoint_ms                                               629550
embedding             [0.18030354380607605, -0.08589434623718262, -0...
video_stem            ase_chronological_bible_translation_in_america...
model                                      asl_finetune_checkpoint_best
window_size_ms                                                      900
file                  CBT-033-ase-2-Passage _ israel fails to conque...
file_stem             CBT-033-ase-2-Passage _ israel fails to conque...
cbt_number                                                           33
cbt_video_type                                                  Passage
title                                        Israel Fails To Conquer Ai
Name: 104865, dtype: object

##### Get query embeddings

In [128]:

query_embeddings = potential_query_embeddings[potential_query_embeddings["window_midpoint_ms"]==629550]
query_embeddings

,window_start_ms,window_end_ms,window_midpoint_ms,embedding,video_stem,model,window_size_ms,file,file_stem,cbt_number,cbt_video_type,title
104865,629100,630000,629550,"[0.18030354380607605, -0.08589434623718262, -0...",ase_chronological_bible_translation_in_america...,asl_finetune_checkpoint_best,900,CBT-033-ase-2-Passage _ israel fails to conque...,CBT-033-ase-2-Passage _ israel fails to conque...,33,Passage,Israel Fails To Conquer Ai


#### Get similarity scores

In [135]:
window_size_ms = 900
example_count = 25

# storage for results
results = []
target_cbt = 33
target_gloss = "BAR"

for (cbt_number, cbt_video_type), file_gt_df in osb_glosses_truth_df.groupby(["cbt_number", "cbt_video_type"]):
    print(file_gt_df.columns)
    title = file_gt_df["title"][0]
    if cbt_number != target_cbt:
        continue
    print(title)
    file_window_embeddings_df = window_embeddings_df[
        (window_embeddings_df["cbt_number"] == cbt_number)
        & (window_embeddings_df["cbt_video_type"] == cbt_video_type)
        & (window_embeddings_df["window_size_ms"] == window_size_ms)
    ]

    if file_window_embeddings_df.empty:
        continue

    file_window_embeddings_array = np.vstack(file_window_embeddings_df["embedding"])

    for sign in file_gt_df["Sign"].unique():
        if sign != target_gloss:
            continue
        # sign_df = isolated_sign_embeddings_df[isolated_sign_embeddings_df["GLOSS"] == sign]
        sign_df = query_embeddings

        if len(sign_df) == 0:
            continue

        # sample consistently for comparability
        n_sample = min(len(sign_df), example_count)
        # sign_df = sign_df.sample(n=example_count, random_state=0)

        sign_df_embeddings = np.vstack(sign_df["embedding"])
        sign_scores = sign_df_embeddings @ file_window_embeddings_array.T  # (n_examples, n_windows)

        # aggregate across the sampled examples
        mean_sign_scores = sign_scores.mean(axis=0)  # (n_windows,)

        # collect results
        for idx, score in enumerate(mean_sign_scores):
            results.append(
                {
                    "cbt_number": cbt_number,
                    "cbt_video_type": cbt_video_type,
                    "Sign": sign,
                    "window_index": idx,
                    "window_midpoint_ms": file_window_embeddings_df["window_midpoint_ms"].iloc[idx],
                    "score": score,
                    "title": title,
                }
            )

# turn into DataFrame for plotting
results_df = pd.DataFrame(results)


Index(['Original Sign', 'Begin Time - hh:mm:ss.ms', 'Begin Time - ss.msec',
       'End Time - hh:mm:ss.ms', 'End Time - ss.msec', 'file', 'duration_sec',
       'midpoint_sec', 'Time (minutes:sec)', 'Sign',
       'ASL Citizen Gloss Repeat Count', 'file_stem', 'cbt_number',
       'cbt_video_type', 'title', 'window_midpoint_ms'],
      dtype='object')
Index(['Original Sign', 'Begin Time - hh:mm:ss.ms', 'Begin Time - ss.msec',
       'End Time - hh:mm:ss.ms', 'End Time - ss.msec', 'file', 'duration_sec',
       'midpoint_sec', 'Time (minutes:sec)', 'Sign',
       'ASL Citizen Gloss Repeat Count', 'file_stem', 'cbt_number',
       'cbt_video_type', 'title', 'window_midpoint_ms'],
      dtype='object')
Index(['Original Sign', 'Begin Time - hh:mm:ss.ms', 'Begin Time - ss.msec',
       'End Time - hh:mm:ss.ms', 'End Time - ss.msec', 'file', 'duration_sec',
       'midpoint_sec', 'Time (minutes:sec)', 'Sign',
       'ASL Citizen Gloss Repeat Count', 'file_stem', 'cbt_number',
       'cbt_vi

In [136]:
results_df

,cbt_number,cbt_video_type,Sign,window_index,window_midpoint_ms,score,title
0,33,Passage,BAR,0,450,67.483411,Israel Fails To Conquer Ai
1,33,Passage,BAR,1,550,67.483411,Israel Fails To Conquer Ai
2,33,Passage,BAR,2,650,67.471576,Israel Fails To Conquer Ai
3,33,Passage,BAR,3,750,67.483411,Israel Fails To Conquer Ai
4,33,Passage,BAR,4,850,67.471576,Israel Fails To Conquer Ai
...,...,...,...,...,...,...,...
7062,33,Passage,BAR,7062,706650,67.471576,Israel Fails To Conquer Ai
7063,33,Passage,BAR,7063,706750,67.483411,Israel Fails To Conquer Ai
7064,33,Passage,BAR,7064,706850,67.471576,Israel Fails To Conquer Ai
7065,33,Passage,BAR,7065,706950,67.483411,Israel Fails To Conquer Ai


#### Plot peaks

In [141]:
import plotly.express as px

# group by cbt_number + cbt_video_type
for (cbt_number, cbt_video_type), group_df in results_df.groupby(["cbt_number", "cbt_video_type"]):
    assert len(group_df["title"].unique()) == 1
    video_title = group_df["title"].unique()[0]
    group_df["Window Midpoint (s)"] = group_df["window_midpoint_ms"]/1000
    fig = px.line(
        group_df,
        x="window_midpoint_ms",
        y="score",
        color="Sign",
        title=f"Mean Sign Scores — {video_title}",
    )
    fig.show()


In [149]:
results_df[results_df["score"]<results_df["score"].max()].sort_values(by="score",ascending=False)

,cbt_number,cbt_video_type,Sign,window_index,window_midpoint_ms,score,title
6291,33,Passage,BAR,6291,629550,124.215594,Israel Fails To Conquer Ai
6293,33,Passage,BAR,6293,629750,124.077140,Israel Fails To Conquer Ai
6290,33,Passage,BAR,6290,629450,121.354792,Israel Fails To Conquer Ai
6294,33,Passage,BAR,6294,629850,120.117744,Israel Fails To Conquer Ai
6289,33,Passage,BAR,6289,629350,117.459449,Israel Fails To Conquer Ai
...,...,...,...,...,...,...,...
1418,33,Passage,BAR,1418,142250,58.158272,Israel Fails To Conquer Ai
5367,33,Passage,BAR,5367,537150,58.078427,Israel Fails To Conquer Ai
877,33,Passage,BAR,877,88150,57.974261,Israel Fails To Conquer Ai
5911,33,Passage,BAR,5911,591550,57.155012,Israel Fails To Conquer Ai


### apply find_peaks

In [59]:
def find_peaks_in_df(
    df: pd.DataFrame,
    group_col="Sign",
    x_col="window_midpoint_ms",
    y_col="score",
    prominence=0.6,
    height=None,
    width=None,  # in number of samples. TODO: set based on step size?
    distance=None,
) -> pd.DataFrame:
    """
    Find peaks in each group and return a DataFrame of peak positions.

    Returns:
    - DataFrame with columns [group_col, x_col, y_col]

    """
    peak_rows = []
    for query_id, group in df.groupby(group_col):
        group = group.sort_values(x_col)
        x = group[x_col].values
        y = group[y_col].values

        # TODO: box plots of properties to investigate
        peaks, properties = find_peaks(y, prominence=prominence, height=height, width=width, distance=distance)
        for i in peaks:
            peak_rows.append(
                {
                    group_col: query_id,
                    x_col: x[i],
                    y_col: y[i],
                }
            )
    return pd.DataFrame(peak_rows, columns=[group_col, x_col, y_col])


### Match Predictions

In [60]:
def match_predictions(
    preds_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    window: int = 1000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Match predictions to ground truth within a ±window (in ms).
    Loops over ground-truth entries to ensure FN are counted correctly even if
    preds_df is empty.
    """
    # At the start of match_predictions
    n_null_preds = preds_df["Sign"].isna().sum()
    n_null_gt = gt_df["Sign"].isna().sum()

    if n_null_preds > 0:
        print(f"Warning: Dropping {n_null_preds} predictions with null Sign")
    if n_null_gt > 0:
        print(f"Warning: Dropping {n_null_gt} ground truth entries with null Sign")

    preds_df = preds_df[preds_df["Sign"].notna()].copy()
    gt_df = gt_df[gt_df["Sign"].notna()].copy()

    preds_out = preds_df.copy()
    preds_out["correct"] = False
    gt_out = gt_df.copy()
    gt_out["found"] = False

    # Track which predictions have been matched to avoid double counting
    matched_preds = set()
    # Loop over GT rows
    for gt_idx, gt_row in gt_df.iterrows():
        gt_sign = gt_row["Sign"]
        gt_ts = gt_row["window_midpoint_ms"]
        # find matching prediction(s) within window
        candidates = preds_out[
            (preds_out["Sign"] == gt_sign)
            & (~preds_out.index.isin(matched_preds))
            & (abs(preds_out["window_midpoint_ms"] - gt_ts) <= window)
        ]
        if not candidates.empty:
            pred_idx = candidates.index[0]  # match first candidate
            preds_out.at[pred_idx, "correct"] = True
            gt_out.at[gt_idx, "found"] = True
            matched_preds.add(pred_idx)
    # --- per-sign metrics ---
    per_sign_records = []
    all_signs = sorted(set(gt_df["Sign"]).union(preds_df.get("Sign", [])))
    for sign in all_signs:
        gt_sign = gt_out[gt_out["Sign"] == sign]
        preds_sign = preds_out[preds_out["Sign"] == sign]

        # Count TP from ground truth side (each GT can only be found once)
        tp = int(gt_sign["found"].sum())
        # FP = predictions that weren't correct
        fp = int((~preds_sign["correct"]).sum())
        # FN = ground truth that wasn't found
        fn = int((~gt_sign["found"]).sum())

        prec = tp / len(preds_sign) if len(preds_sign) > 0 else float("nan")
        rec = tp / len(gt_sign) if len(gt_sign) > 0 else float("nan")
        f1_sign = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        per_sign_records.append(
            {
                "Sign": sign,
                "precision": prec,
                "recall": rec,
                "f1": f1_sign,
                "true_positives": tp,
                "false_positives": fp,
                "false_negatives": fn,
                "n_preds": len(preds_sign),
                "n_gt": len(gt_sign),
            }
        )
    per_sign_df = pd.DataFrame.from_records(per_sign_records).set_index("Sign")
    return preds_out, gt_out, per_sign_df


def aggregate_metrics(per_sign_df: pd.DataFrame) -> dict[str, float]:
    """Aggregate overall precision/recall/F1 from a per-sign stats DataFrame."""
    tp = int(per_sign_df["true_positives"].sum())
    fp = int(per_sign_df["false_positives"].sum())
    fn = int(per_sign_df["false_negatives"].sum())
    n_preds = int(per_sign_df["n_preds"].sum())
    n_gt = int(per_sign_df["n_gt"].sum())

    precision = tp / n_preds if n_preds > 0 else float("nan")
    recall = tp / n_gt if n_gt > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else float("nan")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "n_preds": n_preds,
        "n_gt": n_gt,
    }


In [61]:
def compute_micro_metrics(df: pd.DataFrame) -> dict[str, float]:
    """Compute micro-averaged precision, recall, f1 from per-video metrics DataFrame."""
    required_cols = {"true_positives", "false_positives", "false_negatives"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns {required_cols}")

    total_tp = int(df["true_positives"].sum())
    total_fp = int(df["false_positives"].sum())
    total_fn = int(df["false_negatives"].sum())

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"micro_precision": precision, "micro_recall": recall, "micro_f1": f1}


# WandB Sweep

In [ ]:
# Filter out null signs from ground truth at the start
osb_glosses_truth_df = osb_glosses_truth_df[osb_glosses_truth_df["Sign"].notna()].copy()
n_null_signs = len(osb_glosses_truth_df) - len(osb_glosses_truth_df[osb_glosses_truth_df["Sign"].notna()])
if n_null_signs > 0:
    print(f"Filtered out {n_null_signs} ground truth entries with null Sign")


# --- hyperparameter grids ---
# height_multipliers = [1.0, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.5]
# height_multipliers = list(np.arange(1.0, 1.5, 0.01))
# height_multipliers = list(np.linspace(1.00, 1.5, 200))
# height_multipliers = list(np.linspace(0.00, 1.5, 200))
# height_multipliers = list(np.linspace(1.209, 1.242, 200))
# height_multipliers = list(np.linspace(1.18, 1.28, 200))
# height_multipliers = list(np.linspace(0.8, 1.5, 300))
# height_multipliers = list(np.linspace(1.16, 1.28, 30))
# height_multipliers = list(np.arange(1.21, 1.25, 0.001))
# height_multipliers = list(np.linspace(1.21, 1.23, 100))
# height_multipliers = list(np.random.uniform(low=0.0, high=1.5, size=25))
height_multipliers = list(np.random.uniform(low=0.8, high=1.4, size=25))
# height_multipliers = list(np.random.uniform(low=1.15, high=1.25, size=20))
# random.shuffle(height_multipliers)
# height_multipliers = height_multipliers[:20]

# prominences = [0.5, 0.6, 0.7, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 10.0, 100.0, 1000.0]
# prominences = list(np.arange(0.1, 100.0, 0.1))
# prominences = list(np.logspace(1.2, 10, 100)) # a wide range!
# prominences = list(np.logspace(-3, 2, 100))  # smaller range
# prominences = list(np.logspace(1.1, 1.9, 50))
# prominences = list(np.arange(18.0, 34.0, 0.1))
# prominences = list(np.arange(7.0, 18.0, 0.1))
# prominences = list(np.arange(19.0, 22.0, 0.01))
# prominences = list(np.linspace(0, 22.0, 100))
# prominences = list(np.linspace(20.0, 30.0, 100))
# prominences = list(np.linspace(1.0, 19.0, 100))
# prominences = list(np.linspace(1.0, 30.0, 100))
# prominences = list(np.linspace(1.0, 30.0, 30))
# prominences = list(np.linspace(0.0, 50.0, 200))
# prominences = list(np.linspace(0.0, 30.0, 200))
# prominences = list(np.linspace(0.0, 22.0, 200))
# prominences = list(np.logspace(-3, 2, 200)) # not much point going past 10^2, F1 hits zero around 50
# prominences = list(np.linspace(0.0, 30.0, 100))
# prominences = list(np.linspace(19.0, 22.0, 100))
prominences = list(np.random.uniform(low=0.0, high=50, size=25))
# prominences = list(np.random.uniform(low=0.0, high=30, size=20))
random.shuffle(prominences)
prominences = [*prominences[:25], None]

# widths = [None, 50, 100, 200, 500, 1000, 10000]
# widths = [None, 1, 10, 100, 1000]
# widths = [None, 1, 2]
# widths = list(np.logspace(0.1, 10.0, 50))
# widths = list(np.logspace(0.0, 1.0, 50)) # 1 to 10
# widths = list(np.logspace(0.0, 2.0, 200))  # 1 to 100
# widths = list(np.logspace(-2.0, 0.0, 50)) # 0.01 to 1
# widths = list(np.linspace(0.0, 20.0, 200))
# widths = list(np.linspace(0.0, 50.0, 200))  # hits 0 around here, not much point going higher
# widths = list(np.linspace(0.0, 15.0, 200))
# widths = [*list(np.linspace(0.0, 10.0, 100)), None]
# widths = [*list(np.linspace(0.0, 7.0, 100)), None]
# widths = list(np.linspace(0.0, 6.952, 200))
widths = list(np.random.uniform(low=0.0, high=50, size=30))
# widths = list(np.random.uniform(low=0.0, high=9, size=20))
random.shuffle(widths)
widths = [*widths[:30], None]


# distances = [
#     1,
#     # None,
#     2,
#     3,
#     4,
#     5,
#     10,
#     # 100,
# ]  # Required minimal horizontal distance (>= 1) in samples between neighbouring peaks. Smaller peaks are removed first until the condition is fulfilled for all remaining peaks.
# distances = list(np.logspace(0.1, 3.0, 100))
# distances = list(np.logspace(0.1, 6.0, 200))
# distances = list(np.logspace(0.1, 10.0, 200))
# distances = list(np.logspace(0.1, 10.0, 200))
# distances = list(np.logspace(0.0, 2.0, 20)) # 1 to 100
# distances = list(np.logspace(0.0, 4.0, 200)) # 1 to 10_000
# distances = list(np.logspace(0.0, 5.0, 200))  # 1 to 100_000
# distances = list(np.logspace(0.0, 6.0, 1000))  # 1 to 1_000_000
# distances = list(np.logspace(0.0, 1.0, 50)) # 1 to 10
# distances = list(np.arange(0.0, 10.0, 1))
# distances = list(np.arange(6, 10.0, 0.1))
# distances = list(np.linspace(250.0, 500.0, 50))
# distances = list(np.linspace(1.0, 100.0, 200))
# distances = list(np.linspace(11.0, 20.0, 100))
# distances = [i + 1 for i in range(43)]
# distances = [i + 1 for i in range(11)]  # must be greater than 1, should be an integer
distances = list(np.random.uniform(low=0.0, high=1000,size=40))
random.shuffle(distances)
# distances = distances[:30]

print(f"{height_multipliers=}")
print(f"{prominences=}")
print(f"{distances=}")
print(f"{widths=}")

match_window_ms = 1000
example_counts = [1, 5, 10, 15, 25]
# example_counts = list(range(1, 29))
# example_counts = [25]
random.shuffle(example_counts)
thresholdings = ["all-gloss", "per-gloss"]
# thresholdings = ["all-gloss"]
random.shuffle(thresholdings)

# all available embedding window sizes
available_window_sizes = sorted(window_embeddings_df["window_size_ms"].unique())
# available_window_sizes = [
#     s for s in available_window_sizes if s in [300, 400, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 2000,3000,8000]
# ]
# available_window_sizes = [s for s in available_window_sizes if s in [1000, 1100, 1400]]
available_window_sizes = [s for s in available_window_sizes if s in [550, 650, 750]]
# available_window_sizes = [s for s in available_window_sizes if s in [8000, 3000, 2000]]
# available_window_sizes = [s for s in available_window_sizes if s not in [10000, 9000, 5000]]
# available_window_sizes = [s for s in available_window_sizes if s in [1400]]
# available_window_sizes = [600, 600, 600, 600, 600, 600, *available_window_sizes]
random.shuffle(available_window_sizes)
print(f"{available_window_sizes=}")


runs_per_window_size = 100

for embedding_window_ms in tqdm(available_window_sizes, desc="Embedding window sweep"):
    # --- generate results_df for this embedding window ---
    results = []
    for (cbt_number, cbt_video_type), file_gt_df in osb_glosses_truth_df.groupby(["cbt_number", "cbt_video_type"]):
        title = file_gt_df["title"].iloc[0]
        file_window_embeddings_df = window_embeddings_df[
            (window_embeddings_df["cbt_number"] == cbt_number)
            & (window_embeddings_df["cbt_video_type"] == cbt_video_type)
            & (window_embeddings_df["window_size_ms"] == embedding_window_ms)
        ]
        if file_window_embeddings_df.empty:
            continue

        file_window_embeddings_array = np.vstack(file_window_embeddings_df["embedding"])

        for sign in file_gt_df["Sign"].unique():
            sign_df = isolated_sign_embeddings_df[isolated_sign_embeddings_df["GLOSS"] == sign]
            if sign_df.empty:
                continue

            n_sample = min(len(sign_df), example_count)
            sign_df = sign_df.sample(n=n_sample, random_state=0)
            sign_embeddings = np.vstack(sign_df["embedding"])
            sign_scores = sign_embeddings @ file_window_embeddings_array.T
            mean_scores = sign_scores.mean(axis=0)

            for idx, score in enumerate(mean_scores):
                results.append(
                    {
                        "cbt_number": cbt_number,
                        "cbt_video_type": cbt_video_type,
                        "Sign": sign,
                        "window_index": idx,
                        "window_midpoint_ms": file_window_embeddings_df["window_midpoint_ms"].iloc[idx],
                        "score": score,
                        "title": title,
                    }
                )

    results_df = pd.DataFrame(results)
    if results_df.empty:
        continue
    print(f"Got {len(results_df)} results")

    # --- sweep over hyperparameters, including thresholding ---
    hyperparam_combos = list(
        itertools.product(height_multipliers, prominences, widths, example_counts, thresholdings, distances)
    )

    random.shuffle(hyperparam_combos)
    hyperparam_combos = hyperparam_combos[:runs_per_window_size]

    for height_multiplier, prominence, width, example_count, thresholding, distance in tqdm(
        hyperparam_combos, desc=f"Threshold sweep for {embedding_window_ms}ms"
    ):
        print(height_multiplier, prominence, width, example_count, thresholding, distance)
        print(f"{width=}")
        run = wandb.init(
            project="sign-bibles-matching",
            config={
                "embedding_window_ms": embedding_window_ms,
                "height_multiplier": height_multiplier,
                "prominence": prominence,
                "width": width,
                "match_window_ms": match_window_ms,
                "example_count": example_count,
                "thresholding": thresholding,
                "distance": distance,
            },
            reinit="create_new",
            tags=["metrics_v3", f"osb_glosses_v{osb_glosses_version}"],
        )

        combo_video_metrics = []
        combo_gloss_metrics = []

        for (cbt_number, cbt_video_type), group_df in results_df.groupby(["cbt_number", "cbt_video_type"]):
            file_gt_df = osb_glosses_truth_df[
                (osb_glosses_truth_df["cbt_number"] == cbt_number)
                & (osb_glosses_truth_df["cbt_video_type"] == cbt_video_type)
            ]

            if thresholding == "all-gloss":
                group_mean = group_df["score"].mean()
                peaks_df = find_peaks_in_df(
                    group_df,
                    height=group_mean * height_multiplier,
                    prominence=prominence,
                    width=width,
                    distance=distance,
                )

            elif thresholding == "per-gloss":
                per_gloss_collector = []
                for sign, sign_results_df in group_df.groupby("Sign"):
                    sign_mean = sign_results_df["score"].mean()
                    sign_peaks = find_peaks_in_df(
                        sign_results_df,
                        height=sign_mean * height_multiplier,
                        prominence=prominence,
                        width=width,
                        distance=distance,
                    )
                    per_gloss_collector.append(sign_peaks)
                peaks_df = (
                    pd.concat(per_gloss_collector) if per_gloss_collector else pd.DataFrame(columns=group_df.columns)
                )

            preds_out, gt_out, per_sign_df = match_predictions(peaks_df, file_gt_df, window=match_window_ms)
            video_metrics = aggregate_metrics(per_sign_df)
            video_metrics.update(
                {
                    "cbt_number": cbt_number,
                    "cbt_video_type": cbt_video_type,
                    "thresholding": thresholding,
                }
            )
            combo_video_metrics.append(video_metrics)

            gloss_df = per_sign_df.reset_index().assign(
                cbt_number=cbt_number,
                cbt_video_type=cbt_video_type,
                thresholding=thresholding,
            )
            combo_gloss_metrics.append(gloss_df)

        video_metrics_df = pd.DataFrame(combo_video_metrics)
        gloss_metrics_df = pd.concat(combo_gloss_metrics, ignore_index=True)

        run.log(compute_micro_metrics(gloss_metrics_df))
        run.log({"video_metrics": wandb.Table(dataframe=video_metrics_df)})
        run.log({"gloss_metrics": wandb.Table(dataframe=gloss_metrics_df)})

        # --- Group by Sign and sum the count columns ---
        grouped_metrics = (
            gloss_metrics_df.groupby("Sign")
            .agg(
                {
                    "true_positives": "sum",
                    "false_positives": "sum",
                    "false_negatives": "sum",
                    "n_preds": "sum",
                    "n_gt": "sum",
                }
            )
            .reset_index()
        )
        # --- Compute micro-metrics per Sign ---
        sign_micro_metrics = []
        for _, row in grouped_metrics.iterrows():
            # build a single-row DataFrame with required count columns
            counts_df = pd.DataFrame(
                [
                    {
                        "true_positives": row["true_positives"],
                        "false_positives": row["false_positives"],
                        "false_negatives": row["false_negatives"],
                    }
                ]
            )
            metrics = compute_micro_metrics(counts_df)
            # combine counts and computed metrics
            metrics.update(
                {
                    "Sign": row["Sign"],
                    "true_positives": row["true_positives"],
                    "false_positives": row["false_positives"],
                    "false_negatives": row["false_negatives"],
                    "n_preds": row["n_preds"],
                    "n_gt": row["n_gt"],
                }
            )
            sign_micro_metrics.append(metrics)

        # --- Convert to DataFrame and reorder columns for readability ---
        sign_micro_df = pd.DataFrame(sign_micro_metrics)
        sign_micro_df = sign_micro_df[
            [
                "Sign",
                "micro_precision",
                "micro_recall",
                "micro_f1",
                "true_positives",
                "false_positives",
                "false_negatives",
                "n_preds",
                "n_gt",
            ]
        ]

        # --- Log to W&B ---
        run.log({"grouped_sign_metrics": wandb.Table(dataframe=sign_micro_df)})

        run.finish()


height_multipliers=[np.float64(0.4041911479550624), np.float64(0.08813405146632791), np.float64(0.6922575713695244), np.float64(1.0877431408805576), np.float64(1.12316567866314), np.float64(0.5432705148574062), np.float64(1.2069227473362198), np.float64(0.21492272161129194), np.float64(0.018687739310120077), np.float64(0.007734712404208366), np.float64(1.0996699366324663), np.float64(1.4840825826861388), np.float64(0.17039014365167138), np.float64(0.18814021905784922), np.float64(0.7538409146663538), np.float64(0.8640824739923898), np.float64(0.8600086690074964), np.float64(0.23992495665460079), np.float64(0.929120399352653), np.float64(0.41724336734939693), np.float64(0.7874104393759472), np.float64(0.21120990607499984), np.float64(0.4116447170352413), np.float64(0.4264507178315297), np.float64(0.30401355980158506)]
prominences=[np.float64(45.8762225975946), np.float64(8.939702886456825), np.float64(9.051980250735525), np.float64(42.133058185142986), np.float64(35.08457610099655), np.

Embedding window sweep:   0%|          | 0/3 [00:00<?, ?it/s]

Got 716350 results


Threshold sweep for 650ms:   0%|          | 0/100 [00:00<?, ?it/s]

0.21120990607499984 32.46537388868758 None 25 per-gloss 898.5834963664665
width=None


...

1.4840825826861388 34.5935641588406 28.055429698215757 1 all-gloss 610.5956510994513
width=np.float64(28.055429698215757)


micro_f1,▁
micro_precision,▁
micro_recall,▁
micro_f1,0
micro_precision,0
micro_recall,0


0.18814021905784922 9.051980250735525 33.4256571133189 10 all-gloss 716.5193579935974
width=np.float64(33.4256571133189)


micro_f1,▁
micro_precision,▁
micro_recall,▁
micro_f1,0.00342
micro_precision,0.03704
micro_recall,0.00179


1.0996699366324663 8.744456591625527 46.29054622563727 25 all-gloss 982.5269769560277
width=np.float64(46.29054622563727)
